# 1. Download the sentiment reviews dataset

In [87]:
import pandas as pd
import numpy as np
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

In [88]:
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)

True

In [89]:
df = pd.read_csv('sentiment_reviews.csv')
df = df.dropna(subset=['review_text', 'sentiment'])

df['sentiment'] = df['sentiment'].map({'positive': 1, 'negative': 0, 'neutral': 0})

df = df.dropna(subset=['sentiment'])

# 2. Preprocess text using NLTK or regex (remove HTML tags, punctuation, stopwords)

In [90]:
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def clean_text(text):
    text = text.lower()
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r'[^\w\s]', '', text)

    # Tokenize, remove stopwords, and lemmatize
    words = text.split()
    words = [lemmatizer.lemmatize(word) for word in words if word not in stop_words]
    return ' '.join(words)

df['cleaned_review'] = df['review_text'].apply(clean_text)

# 3. Convert text to numerical vectors using TfidfVectorizer(max_features=5000)

In [91]:
X_train = df['cleaned_review']
y_train = df['sentiment']

In [92]:
tfidf = TfidfVectorizer(max_features=5000)
X_train_tfidf = tfidf.fit_transform(X_train)

# 4. Train MultinomialNB and LogisticRegression

In [93]:
log_reg = LogisticRegression(max_iter=1000, random_state=42)
log_reg.fit(X_train_tfidf, y_train)
nb_model = MultinomialNB()
nb_model.fit(X_train_tfidf, y_train)

print("Models trained successfully!")

Models trained successfully!


# 5. Create a predict_sentiment() function that tests live custom user input phrases

In [94]:
def predict_sentiment(custom_text):
    cleaned = clean_text(custom_text)

    vectorized = tfidf.transform([cleaned])

    prediction = log_reg.predict(vectorized)[0]
    probability = log_reg.predict_proba(vectorized)[0]

    sentiment = "Positive" if prediction == 1 else "Negative"
    confidence = probability[1] if prediction == 1 else probability[0]

    print(f"Review: '{custom_text}'")
    print(f"Predicted Sentiment: {sentiment} (Confidence: {confidence:.1%})\n")

In [95]:
predict_sentiment("This product was absolutely amazing, highly recommend it!")
predict_sentiment("Terrible experience. The quality was awful and it broke on day one.")

Review: 'This product was absolutely amazing, highly recommend it!'
Predicted Sentiment: Positive (Confidence: 56.8%)

Review: 'Terrible experience. The quality was awful and it broke on day one.'
Predicted Sentiment: Negative (Confidence: 53.7%)

